In [ ]:
import os
from pathlib import Path
import sys

import librosa
from matplotlib import colors
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import IPython.display as ipd
import tensorflow as tf
import tensorflow.keras as K
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Conv2DTranspose
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras.layers import Reshape
from tensorflow.keras.layers import UpSampling2D
from tensorflow.keras.models import model_from_json
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import RMSprop
from tqdm.notebook import tqdm

In [ ]:
RANDOM_SEED = 13131313
np.random.seed(RANDOM_SEED)

ROOT_DIR = os.path.abspath('..')  # ROOT_DIR = Path(__file__).parents[1].resolve()
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)
RESEARCH_DIR = ROOT_DIR / Path('researches')

TRAIN_DATA_PATH = ROOT_DIR / Path('data/processed_train_data_rol_len64_rows4/binary')

The idea is to transform audio to image-like data and use DCGAN.

To use GAN or DCGAN audio needs to be processed. It was tested that converting to Mel-Spectrogram and backward works really bad with NES melodies. So Separated Score Format was chosen.

# Generator

In [ ]:
generator = Sequential()

# Dense
generator.add(Dense(32 * 1 * 256, input_dim=NOISE_INPUT_DIM))
generator.add(BatchNormalization(momentum=BATCH_NORMALIZATION_MOMENTUM))
generator.add(Activation('relu'))
generator.add(Reshape((32, 1, 256)))
generator.add(Dropout(DROPOUT))
# 32 * 1 * 256

# Upsampling (instead of fractionally-strided convolution) and transposed convolution 1
generator.add(UpSampling2D())
generator.add(Conv2DTranspose(128, KERNEL_SIZE, padding='same'))
generator.add(BatchNormalization(momentum=BATCH_NORMALIZATION_MOMENTUM))
generator.add(Activation('relu'))
# 64 * 2 * 128

# Upsampling (instead of fractionally-strided convolution) and transposed convolution 2
generator.add(UpSampling2D())
generator.add(Conv2DTranspose(64, KERNEL_SIZE, padding='same'))
generator.add(BatchNormalization(momentum=BATCH_NORMALIZATION_MOMENTUM))
generator.add(Activation('relu'))
# 128 * 4 * 64

# Upsampling (instead of fractionally-strided convolution) and transposed convolution 3
generator.add(Conv2DTranspose(1, KERNEL_SIZE, padding='same'))
generator.add(BatchNormalization(momentum=BATCH_NORMALIZATION_MOMENTUM))
generator.add(Activation('relu'))
# 128 * 4 * 32

generator.add(Conv2DTranspose(1, KERNEL_SIZE, padding='same'))
generator.add(Activation('sigmoid'))
# 128 * 4 * 1

generator.summary()

# Discriminator

In [ ]:
discriminator = Sequential()
input_shape = (SAMPLE_LEN, SAMPLE_WIDTH, 1)

discriminator.add(Conv2D(64, KERNEL_SIZE, strides=2, input_shape=input_shape,
                         padding='same', activation=LeakyReLU(alpha=LEAKY_ALPHA)))
discriminator.add(Dropout(DROPOUT))
discriminator.add(Conv2D(128, KERNEL_SIZE, strides=2, padding='same', activation=LeakyReLU(alpha=LEAKY_ALPHA)))
discriminator.add(Dropout(DROPOUT))
discriminator.add(Conv2D(256, KERNEL_SIZE, strides=2, padding='same', activation=LeakyReLU(alpha=LEAKY_ALPHA)))
discriminator.add(Dropout(DROPOUT))
discriminator.add(Conv2D(512, KERNEL_SIZE, strides=1, padding='same', activation=LeakyReLU(alpha=LEAKY_ALPHA)))
discriminator.add(Dropout(DROPOUT))

discriminator.add(Flatten())
discriminator.add(Dense(1))
discriminator.add(Activation('sigmoid'))
discriminator.summary()

# Discriminator Model

In [ ]:
discriminator_model= Sequential()
discriminator_model.add(discriminator)
discriminator_model.compile(loss='binary_crossentropy',
                            optimizer=RMSprop(lr=D_LEARNING_RATE, clipvalue=1.0, decay=6e-8),metrics=['accuracy'])

# Adversarial Model

In [ ]:
adversarial_model = Sequential()
adversarial_model.add(generator)
adversarial_model.add(discriminator)
adversarial_model.compile(loss='binary_crossentropy',
                          optimizer=RMSprop(lr=A_LEARNING_RATE, clipvalue=1.0, decay=3e-8), metrics=['accuracy'])

---

# Training

https://www.tensorflow.org/tutorials/generative/dcgan

https://tfs.rubius.com/RubiusProjects/Polyphemus/_git/cyclop-server?path=%2Fcomponents%2Fsiamese_network%2Ftrain_model.py

---

https://www.tensorflow.org/tutorials/customization/custom_training_walkthrough

https://www.tensorflow.org/guide/keras/writing_a_training_loop_from_scratch

https://www.tensorflow.org/guide/keras/custom_callback

In [ ]:
class 